# Clothes segmentation: try the model on your own photos

Upload a photo of a person; the final model segments it and shows:

1. the original, 2. the segmentation in colours (all 15 classes), 3. the colours over the photo,
4. **only the clothes, everything else black**, 5. the clothes as a white mask,
6. the clothes in their class colours, everything else black, 7. the person with the background removed,
8. only the body (hair, face, arms, legs), and then **each garment found, alone on black**.

Every picture is also saved to Drive (`clothseg/reports/my_photos/<photo name>/`), together with `clothes_transparent.png` (the clothes on a see-through background) and `label_mask.png`.

After each photo the notebook asks whether you want to upload another one.

A **T4 GPU is enough**. Run the cells from top to bottom; cell 3 asks for `clothseg_code.zip` only if Drive has an older code version. "Clothes" means the 10 clothing and accessory classes: hat, sunglasses, upper-clothes, skirt, pants, dress, belt, shoes, bag, scarf.

## 1. Settings

In [ ]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "19"
SESSION_BUDGET_MIN = 600   # used by a helper in cell 2; nothing is trained here

RUN_NAME = "final_segformer_b3_e30"   # the final model
CHECKPOINT = f"{DRIVE_ROOT}/runs/{RUN_NAME}/checkpoints/best.pt"
FLIP = True        # flip averaging, as in the reported results; False is about twice as fast
MIN_AREA = 0.002   # a garment smaller than 0.2% of the photo is not shown on its own
DOWNLOAD = False   # True: also download a zip of each photo's pictures to your computer

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)

## 5. Load the model

In [ ]:
import sys
sys.path.insert(0, f"{CODE_DIR}/src")
from clothseg.inference import Segmenter, load_image
from clothseg.labels import garment_ids

assert os.path.exists(CHECKPOINT), f"Model not found: {CHECKPOINT}. Check RUN_NAME in cell 1."
seg = Segmenter(CHECKPOINT, flip_tta=FLIP)
print(f"model loaded: {seg.cfg.model}, epoch {seg.epoch}, {seg.num_classes} classes | flip averaging {'on' if FLIP else 'off'} | {seg.device}")

## 6. Helper functions (run once)

In [ ]:
import io, math, re, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

OUT_ROOT = f"{DRIVE_ROOT}/reports/my_photos"
GARMENTS = garment_ids(seg.num_classes)                    # hat, sunglasses, upper-clothes, skirt, pants, dress, belt, shoes, bag, scarf
BODY = [i for i, n in enumerate(seg.names) if n in ("hair", "face", "arms", "legs")]


def keep(image, region):
    """The photo where region is True, black everywhere else."""
    return np.where(region[..., None], image, 0).astype(np.uint8)


def views(image, mask):
    """All output pictures for one photo, as (title, RGB array) pairs."""
    clothes = np.isin(mask, GARMENTS)
    colours = seg.colorize(mask)
    out = [
        ("1. Original", image),
        ("2. Segmentation, all classes", colours),
        ("3. Overlay on the photo", seg.overlay(image, mask)),
        ("4. Clothes only, rest black", keep(image, clothes)),
        ("5. Clothes mask (white)", np.repeat(clothes[..., None], 3, axis=2).astype(np.uint8) * 255),
        ("6. Clothes in class colours, rest black", keep(colours, clothes)),
        ("7. Person, background removed", keep(image, mask != 0)),
        ("8. Body only: hair, face, arms, legs", keep(image, np.isin(mask, BODY))),
    ]
    for cid in GARMENTS:
        region = mask == cid
        if region.mean() >= MIN_AREA:
            out.append((f"{len(out) + 1}. Only the {seg.names[cid]}", keep(image, region)))
    return out


def show(name, image, mask, pictures):
    """Draw all pictures in a grid with a colour legend of the classes found."""
    cols = 4
    rows = math.ceil(len(pictures) / cols)
    h, w = image.shape[:2]
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * (5 * h / w + 0.45) + 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, pic) in zip(axes.flat, pictures):
        ax.imshow(pic)
        ax.set_title(title, fontsize=11)
    present = [i for i in np.unique(mask) if i != 0]
    handles = [mpatches.Patch(color=seg.palette[i] / 255, label=seg.names[i]) for i in present]
    fig.legend(handles=handles, loc="lower center", ncol=min(8, max(1, len(handles))), fontsize=11, frameon=False)
    fig.suptitle(name, fontsize=14, y=1.0)
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    plt.show()


def process(name, data):
    """Segment one uploaded photo, show every output, save them to Drive. Returns the output folder."""
    image = load_image(io.BytesIO(data))
    mask = seg.predict(image)
    pictures = views(image, mask)
    show(name, image, mask, pictures)

    counts = np.bincount(mask.ravel(), minlength=seg.num_classes)
    found = sorted(((100 * counts[i] / mask.size, seg.names[i]) for i in range(1, seg.num_classes) if counts[i] >= 0.0005 * mask.size), reverse=True)
    print("share of the photo per class:  " + ",  ".join(f"{n} {p:.1f}%" for p, n in found))
    missing = [seg.names[i] for i in GARMENTS if counts[i] and counts[i] / mask.size < MIN_AREA]
    if missing:
        print(f"not shown on their own (under {100 * MIN_AREA:.1f}% of the photo):", ", ".join(missing))

    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", name.rsplit(".", 1)[0]).strip("_") or "photo"
    out_dir = f"{OUT_ROOT}/{stem}"
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir)
    for k, (title, pic) in enumerate(pictures, 1):
        slug = re.sub(r"[^a-z0-9]+", "_", title.split(". ", 1)[-1].lower()).strip("_")
        Image.fromarray(pic).save(f"{out_dir}/{k:02d}_{slug}.png")
    clothes = np.isin(mask, GARMENTS)
    Image.fromarray(np.dstack([image, clothes.astype(np.uint8) * 255])).save(f"{out_dir}/clothes_transparent.png")
    label_png = Image.frombytes("P", (mask.shape[1], mask.shape[0]), np.ascontiguousarray(mask, dtype=np.uint8).tobytes())
    label_png.putpalette(seg.palette.flatten().tolist())
    label_png.save(f"{out_dir}/label_mask.png")
    print(f"saved {len(pictures) + 2} pictures to {out_dir} (including clothes_transparent.png, a cut-out with a see-through background)")
    return out_dir

## 7. Upload photos and see the results

Choose one or more photos in the upload box. After they are shown, type `y` to upload another photo, or anything else to stop. If a photo looks wrong, the reasons the model can fail are listed in the report (report_latex/report.pdf), sections 5.2 and 5.3.

In [ ]:
from google.colab import files

done = 0
while True:
    print()
    print("Choose one or more photos of a person (JPG or PNG).")
    try:
        uploaded = files.upload()
    except Exception as exc:  # the upload box was cancelled or the browser lost the connection
        print(f"upload stopped ({type(exc).__name__})")
        uploaded = {}
    for name, data in uploaded.items():
        try:
            out_dir = process(name, data)
        except Exception as exc:
            print(f"{name}: could not be segmented ({type(exc).__name__}: {exc}). Is it an image file?")
            continue
        done += 1
        if DOWNLOAD:
            shutil.make_archive(out_dir, "zip", out_dir)
            files.download(out_dir + ".zip")
    if not uploaded:
        print("no photo was received")
    answer = input("Upload another photo? Type y for yes, anything else to stop: ").strip().lower()
    if answer not in ("y", "yes"):
        break
print(f"Finished. {done} photo(s) segmented. All results are on Drive in {OUT_ROOT}")